<a href="https://colab.research.google.com/github/Harshada900/pyspark-practise/blob/main/3_Transformation_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Session2") \
    .config("spark.sql.shuffle.partitions", "5") \
    .getOrCreate()

In [3]:
data = [("Raj",   "India", "Engineering", 50000, None),
        ("Priya", "USA",   "Marketing",   70000, 28),
        ("Amit",  "India", "Engineering", 60000, 25),
        ("Sara",  "UK",    "HR",          80000, 32),
        ("Karan", "India", "Marketing",   45000, 27)]

columns = ["name", "country", "department", "salary", "age"]

In [4]:
df = spark.createDataFrame(data, columns)

In [5]:
#1 select - selects specific columns

df.select("name", "salary").show()


+-----+------+
| name|salary|
+-----+------+
|  Raj| 50000|
|Priya| 70000|
| Amit| 60000|
| Sara| 80000|
|Karan| 45000|
+-----+------+



In [6]:
#other ways to select
from pyspark.sql.functions import col
df.select(col("name"), col("salary")).show()

+-----+------+
| name|salary|
+-----+------+
|  Raj| 50000|
|Priya| 70000|
| Amit| 60000|
| Sara| 80000|
|Karan| 45000|
+-----+------+



In [7]:
df.select("*").show()   #shows all data

+-----+-------+-----------+------+----+
| name|country| department|salary| age|
+-----+-------+-----------+------+----+
|  Raj|  India|Engineering| 50000|NULL|
|Priya|    USA|  Marketing| 70000|  28|
| Amit|  India|Engineering| 60000|  25|
| Sara|     UK|         HR| 80000|  32|
|Karan|  India|  Marketing| 45000|  27|
+-----+-------+-----------+------+----+



In [8]:
#select with an expression

df.select("salary", (col("salary")*1.1).alias("new_salary")).show()   #(col("salary")*1.1) is an expression

+------+-----------------+
|salary|       new_salary|
+------+-----------------+
| 50000|55000.00000000001|
| 70000|          77000.0|
| 60000|          66000.0|
| 80000|          88000.0|
| 45000|49500.00000000001|
+------+-----------------+



In [9]:
#filter/where - bothe are identical and does exactly the same thing

#filter using string expression -> sql style

df.filter("salary>55000").show()

+-----+-------+-----------+------+---+
| name|country| department|salary|age|
+-----+-------+-----------+------+---+
|Priya|    USA|  Marketing| 70000| 28|
| Amit|  India|Engineering| 60000| 25|
| Sara|     UK|         HR| 80000| 32|
+-----+-------+-----------+------+---+



In [10]:
#filter using col() -> python style

df.filter(col("salary")>55000).show()

+-----+-------+-----------+------+---+
| name|country| department|salary|age|
+-----+-------+-----------+------+---+
|Priya|    USA|  Marketing| 70000| 28|
| Amit|  India|Engineering| 60000| 25|
| Sara|     UK|         HR| 80000| 32|
+-----+-------+-----------+------+---+



In [11]:
#where identical to filter!

df.where("salary>55000").show()

+-----+-------+-----------+------+---+
| name|country| department|salary|age|
+-----+-------+-----------+------+---+
|Priya|    USA|  Marketing| 70000| 28|
| Amit|  India|Engineering| 60000| 25|
| Sara|     UK|         HR| 80000| 32|
+-----+-------+-----------+------+---+



### **Multiple Conditions**

In [12]:
#AND -> &

df.filter((col("salary")>60000) & (col("country")=='USA')).show()

+-----+-------+----------+------+---+
| name|country|department|salary|age|
+-----+-------+----------+------+---+
|Priya|    USA| Marketing| 70000| 28|
+-----+-------+----------+------+---+



In [13]:
#OR -> |


df.filter((col("country")=='USA') | (col("country")=='UK')).show()

+-----+-------+----------+------+---+
| name|country|department|salary|age|
+-----+-------+----------+------+---+
|Priya|    USA| Marketing| 70000| 28|
| Sara|     UK|        HR| 80000| 32|
+-----+-------+----------+------+---+



In [14]:
#NOT -> !=

df.filter(col("country")!='USA').show()

+-----+-------+-----------+------+----+
| name|country| department|salary| age|
+-----+-------+-----------+------+----+
|  Raj|  India|Engineering| 50000|NULL|
| Amit|  India|Engineering| 60000|  25|
| Sara|     UK|         HR| 80000|  32|
|Karan|  India|  Marketing| 45000|  27|
+-----+-------+-----------+------+----+



## **Checking Nulls**

In [15]:
df.filter(col("age").isNull()).show()

+----+-------+-----------+------+----+
|name|country| department|salary| age|
+----+-------+-----------+------+----+
| Raj|  India|Engineering| 50000|NULL|
+----+-------+-----------+------+----+



In [16]:
df.filter(col("age").isNotNull()).show()

+-----+-------+-----------+------+---+
| name|country| department|salary|age|
+-----+-------+-----------+------+---+
|Priya|    USA|  Marketing| 70000| 28|
| Amit|  India|Engineering| 60000| 25|
| Sara|     UK|         HR| 80000| 32|
|Karan|  India|  Marketing| 45000| 27|
+-----+-------+-----------+------+---+



In [17]:
# 3 withColumn -> add or modify a column

df.withColumn("salary_k", col("salary")/1000).show()

+-----+-------+-----------+------+----+--------+
| name|country| department|salary| age|salary_k|
+-----+-------+-----------+------+----+--------+
|  Raj|  India|Engineering| 50000|NULL|    50.0|
|Priya|    USA|  Marketing| 70000|  28|    70.0|
| Amit|  India|Engineering| 60000|  25|    60.0|
| Sara|     UK|         HR| 80000|  32|    80.0|
|Karan|  India|  Marketing| 45000|  27|    45.0|
+-----+-------+-----------+------+----+--------+



In [18]:
#modifying existing column

df.withColumn("salary", col("salary")*1.1).show()

+-----+-------+-----------+-----------------+----+
| name|country| department|           salary| age|
+-----+-------+-----------+-----------------+----+
|  Raj|  India|Engineering|55000.00000000001|NULL|
|Priya|    USA|  Marketing|          77000.0|  28|
| Amit|  India|Engineering|          66000.0|  25|
| Sara|     UK|         HR|          88000.0|  32|
|Karan|  India|  Marketing|49500.00000000001|  27|
+-----+-------+-----------+-----------------+----+



In [19]:
df.show()       #no change in original df bcz df is immutable so it returns new df

+-----+-------+-----------+------+----+
| name|country| department|salary| age|
+-----+-------+-----------+------+----+
|  Raj|  India|Engineering| 50000|NULL|
|Priya|    USA|  Marketing| 70000|  28|
| Amit|  India|Engineering| 60000|  25|
| Sara|     UK|         HR| 80000|  32|
|Karan|  India|  Marketing| 45000|  27|
+-----+-------+-----------+------+----+



In [20]:
#add a column with existing value

from pyspark.sql.functions import lit

df.withColumn("currency", lit("INR")).show()

+-----+-------+-----------+------+----+--------+
| name|country| department|salary| age|currency|
+-----+-------+-----------+------+----+--------+
|  Raj|  India|Engineering| 50000|NULL|     INR|
|Priya|    USA|  Marketing| 70000|  28|     INR|
| Amit|  India|Engineering| 60000|  25|     INR|
| Sara|     UK|         HR| 80000|  32|     INR|
|Karan|  India|  Marketing| 45000|  27|     INR|
+-----+-------+-----------+------+----+--------+



In [24]:
# chaning currency country wise

from pyspark.sql.functions import when

df.withColumn("currency",
when(col("country")=='India', "INR")
.when(col("country")=='USA','USD')
.when(col("country")=='UK', 'GBP')
).show()


+-----+-------+-----------+------+----+--------+
| name|country| department|salary| age|currency|
+-----+-------+-----------+------+----+--------+
|  Raj|  India|Engineering| 50000|NULL|     INR|
|Priya|    USA|  Marketing| 70000|  28|     USD|
| Amit|  India|Engineering| 60000|  25|     INR|
| Sara|     UK|         HR| 80000|  32|     GBP|
|Karan|  India|  Marketing| 45000|  27|     INR|
+-----+-------+-----------+------+----+--------+



In [25]:
# drop - removes  column

#drop one column

df.drop("age").show()

+-----+-------+-----------+------+
| name|country| department|salary|
+-----+-------+-----------+------+
|  Raj|  India|Engineering| 50000|
|Priya|    USA|  Marketing| 70000|
| Amit|  India|Engineering| 60000|
| Sara|     UK|         HR| 80000|
|Karan|  India|  Marketing| 45000|
+-----+-------+-----------+------+



In [27]:
#drop multiple columns

df.drop("age", "department").show()

+-----+-------+------+
| name|country|salary|
+-----+-------+------+
|  Raj|  India| 50000|
|Priya|    USA| 70000|
| Amit|  India| 60000|
| Sara|     UK| 80000|
|Karan|  India| 45000|
+-----+-------+------+



In [28]:
#but in real life we dont write code in single line!

result = df.filter(col("country")=='India').select("name", "age", "salary").withColumn("salary_k", col("salary")/1000).drop("salary")

In [29]:
result.show()

+-----+----+--------+
| name| age|salary_k|
+-----+----+--------+
|  Raj|NULL|    50.0|
| Amit|  25|    60.0|
|Karan|  27|    45.0|
+-----+----+--------+

